In [ ]:
import pandas as pd

df_train = pd.read_csv("train.csv")
# test = pd.read_csv("test.csv")

print(df_train.shape)
df_train.head()


In [ ]:
df_train.dtypes

In [ ]:
df_train.isnull().sum()

In [ ]:
# %%
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(df_train["ASI_category"])   # One-time encoding

# Split features and target
X = df_train.drop(["ID", "ASI_category"], axis=1)

# Split into train/validation sets (stratified for balance)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.05, random_state=42, stratify=y
)

print("Training set:", X_train.shape, "Validation set:", X_val.shape)
print("Classes:", le.classes_)


In [ ]:
# # %% [Train XGBoost with tuned parameters and evaluate on Train & Validation sets]

# from xgboost import XGBClassifier
# from sklearn.metrics import accuracy_score, f1_score, mean_squared_error
# import numpy as np

# # Define tuned XGBoost model
# best_xgb = XGBClassifier(
#     n_estimators=300,
#     learning_rate=0.01, 
#     max_depth=5, 
#     subsample=0.7, 
#     colsample_bytree=0.6, 
#     gamma=1.39, 
#     min_child_weight=3,
#     random_state=42,
#     use_label_encoder=False,
#     eval_metric="mlogloss",
#     n_jobs=-1
# )

# # Train model
# best_xgb.fit(X_train, y_train)

# # Predictions for both train and validation sets
# y_pred_train = best_xgb.predict(X_train)
# y_pred_val = best_xgb.predict(X_val)

# # ? Compute metrics for train and validation sets
# metrics = {}

# for label, y_true, y_pred in [
#     ("Train", y_train, y_pred_train),
#     ("Validation", y_val, y_pred_val)
# ]:
#     acc = accuracy_score(y_true, y_pred)
#     f1 = f1_score(y_true, y_pred, average="macro")
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     metrics[label] = {"Accuracy": acc, "F1": f1, "RMSE": rmse}

# # Display results nicely
# print("? Model Evaluation Metrics:")
# for dataset, scores in metrics.items():
#     print(f"\n?? {dataset} Set:")
#     print(f"Accuracy : {scores['Accuracy']:.4f}")
#     print(f"F1 Score : {scores['F1']:.4f}")
#     print(f"RMSE     : {scores['RMSE']:.4f}")


In [ ]:
# %% [Optuna Hyperparameter Tuning for XGBoost - Clean Version]

import optuna
from optuna.samplers import TPESampler
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np


def objective(trial):
    # --- Define optimized parameter search space ---
    params = {
        # Core parameters
        "n_estimators": trial.suggest_int("n_estimators", 200, 700),
        "eta": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 2.5),

        # Subsampling for regularization
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.5, 1.0),

        # Regularization
        "lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),
        "alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),

        # Booster and objective
        "booster": "gbtree",  # fixed to gbtree (no dart)
        "objective": "multi:softprob",
        "eval_metric": "mlogloss",

        # Misc
        "num_class": len(np.unique(y_train)),
        "random_state": 42,
        "verbosity": 0,
        "nthread": -1,
    }

    # --- Split data into sub-train / validation for evaluation ---
    X_sub, X_hold, y_sub, y_hold = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
    )

    dtrain = xgb.DMatrix(X_sub, label=y_sub)
    dvalid = xgb.DMatrix(X_hold, label=y_hold)

    # --- Train model with early stopping ---
    evals = [(dtrain, "train"), (dvalid, "val")]
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1500,
        evals=evals,
        early_stopping_rounds=50,
        verbose_eval=False
    )

    # --- Predict and evaluate ---
    preds = np.argmax(model.predict(dvalid), axis=1)
    f1 = f1_score(y_hold, preds, average="macro")

    return f1


# --- Run Optuna Study ---
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=200, show_progress_bar=True)

print("\n? Best XGBoost Parameters:")
print(study.best_params)
print(f"Best Macro F1 Score: {study.best_value:.4f}")
